[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sausterm/omniscience-research/blob/main/pcet_engine/notebooks/pcet_demo.ipynb)
[![View on GitHub](https://img.shields.io/badge/GitHub-View%20Source-blue?logo=github)](https://github.com/sausterm/omniscience-research/blob/main/pcet_engine/notebooks/pcet_demo.ipynb)

# PCET Rate Prediction Engine
## Vibronic Rate Calculation from Marcus Parameters

**OmniSciences LLC** | [omnisciences.io](https://omnisciences.io)

---

This notebook demonstrates the OmniSciences PCET (Proton-Coupled Electron Transfer) Rate Prediction API,
which computes hydrogen and deuterium transfer rates, kinetic isotope effects (KIE), and activation
energies from Marcus theory parameters.

PCET reactions are central to enzyme catalysis (soybean lipoxygenase, alcohol dehydrogenase, glucose oxidase),
energy conversion (photosystem II, hydrogenases), and electrochemistry (fuel cells, CO2 reduction).
The API implements Marcus theory, single-mode vibronic theory, and multi-mode vibronic theory with
full uncertainty quantification.

> **API Key**: Contact sloan@omnisciences.org for access.

In [ ]:
import requests
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12

# -- Configuration ---------------------------------------------------------
API_URL = "http://pcet-engine-prod-1031739699.us-east-1.elb.amazonaws.com"
API_KEY = "your-api-key-here"  # Get yours at omnisciences.io
# --------------------------------------------------------------------------

HEADERS = {"X-API-Key": API_KEY, "Content-Type": "application/json"}


def api_get(endpoint):
    """GET from the PCET API and return the JSON response."""
    url = f"{API_URL}{endpoint}"
    resp = requests.get(url, headers=HEADERS, timeout=30)
    resp.raise_for_status()
    return resp.json()


def api_post(endpoint, payload):
    """POST to the PCET API and return the JSON response."""
    url = f"{API_URL}{endpoint}"
    resp = requests.post(url, json=payload, headers=HEADERS, timeout=60)
    resp.raise_for_status()
    return resp.json()


# Quick connectivity check
try:
    info = api_get("/health")
    print(f"API status:  {info.get('status', 'OK')}")
    for k, v in info.items():
        if k != 'status':
            print(f"  {k}: {v}")
except Exception as e:
    print(f"Cannot reach API at {API_URL}")
    print(f"Error: {e}")

## 1. Benchmark Systems

The PCET engine has been validated against **15 benchmark enzyme systems** spanning
lipoxygenases, oxidases, dehydrogenases, and other PCET-active enzymes. Each benchmark
includes experimentally measured KIE values and the Marcus parameters needed to reproduce them.

In [ ]:
benchmarks = api_get("/v1/benchmarks")

# Display as formatted table
print(f"{'Enzyme':<30} {'KIE (exp)':>10} {'KIE (pred)':>11} {'Error %':>9}")
print("=" * 62)
for b in benchmarks:
    name = b.get('name', b.get('enzyme', 'Unknown'))
    kie_exp = b.get('kie_experimental', b.get('KIE_exp', None))
    kie_pred = b.get('kie_predicted', b.get('KIE_pred', None))
    if kie_exp is not None and kie_pred is not None:
        err = abs(kie_pred - kie_exp) / kie_exp * 100
        print(f"{name:<30} {kie_exp:>10.1f} {kie_pred:>11.1f} {err:>8.1f}%")
    else:
        print(f"{name:<30} {'--':>10} {'--':>11} {'--':>9}")

# Bar chart: experimental vs predicted KIE
names = [b.get('name', b.get('enzyme', f'System {i}')) for i, b in enumerate(benchmarks)]
kie_exps = [b.get('kie_experimental', b.get('KIE_exp', 0)) for b in benchmarks]
kie_preds = [b.get('kie_predicted', b.get('KIE_pred', 0)) for b in benchmarks]

# Show top systems by KIE magnitude
order = np.argsort(kie_exps)[::-1]
top_n = min(12, len(benchmarks))
idx = order[:top_n]

fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(top_n)
width = 0.35

ax.bar(x - width/2, [kie_exps[i] for i in idx], width,
       label='Experimental', color='#2980b9', edgecolor='black', linewidth=0.5)
ax.bar(x + width/2, [kie_preds[i] for i in idx], width,
       label='Predicted', color='#e74c3c', edgecolor='black', linewidth=0.5)

ax.set_xlabel('Enzyme System')
ax.set_ylabel('KIE (k_H / k_D)')
ax.set_title('Benchmark Validation: Experimental vs Predicted KIE', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([names[i] for i in idx], rotation=45, ha='right', fontsize=9)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 2. Single Rate Calculation

The core API endpoint accepts Marcus theory parameters and returns H/D transfer rates, KIE,
and activation energy. The key inputs are:

| Parameter | Symbol | Units | Description |
|-----------|--------|-------|-------------|
| `V_el` | V_el | kcal/mol | Electronic coupling |
| `delta_G` | dG | kcal/mol | Driving force (reaction free energy) |
| `lambda_reorg` | lambda | kcal/mol | Reorganization energy |
| `omega_H` | omega_H | cm-1 | Proton vibrational frequency |
| `d_DA` | d_DA | Angstrom | Donor-acceptor distance |

We compute rates for **soybean lipoxygenase (SLO-1)**, the most-studied PCET enzyme,
using all three methods: Marcus, vibronic (single mode), and vibronic (multi-mode).

In [ ]:
# SLO-1 wild-type parameters
slo1_params = {
    "V_el": 1.0,
    "delta_G": -5.4,
    "lambda_reorg": 19.0,
    "omega_H": 2900,
    "d_DA": 2.7,
    "temperature": 303.0
}

methods = ["marcus", "vibronic_single", "vibronic_multi"]
results = {}

for method in methods:
    params = {**slo1_params, "method": method}
    results[method] = api_post("/v1/rate/parameters", params)

# Display comparison
print(f"{'Quantity':<25} {'Marcus':>15} {'Vibronic (1)':>15} {'Vibronic (N)':>15}")
print("=" * 72)
for key, label in [("k_H", "k_H (s^-1)"), ("k_D", "k_D (s^-1)"),
                    ("KIE", "KIE"), ("E_a", "E_a (kcal/mol)")]:
    vals = []
    for method in methods:
        v = results[method].get(key, None)
        if v is not None:
            if key in ("k_H", "k_D"):
                vals.append(f"{v:.2e}")
            else:
                vals.append(f"{v:.2f}")
        else:
            vals.append("--")
    print(f"{label:<25} {vals[0]:>15} {vals[1]:>15} {vals[2]:>15}")

# Show tunneling contribution if available
for method in methods:
    tc = results[method].get('tunneling_contribution', None)
    if tc is not None:
        print(f"\nTunneling contribution ({method}): {tc:.1%}")

## 3. Parameter Sensitivity

How sensitive is the KIE to each Marcus parameter? We sweep two key variables:

- **Donor-acceptor distance (d_DA)**: Controls proton tunneling probability. Longer distances
  suppress tunneling, reducing KIE.
- **Reorganization energy (lambda)**: The energy penalty for nuclear rearrangement. Higher
  lambda increases the classical barrier, amplifying the tunneling advantage of H over D.

In [ ]:
# Sweep donor-acceptor distance
d_DA_values = np.linspace(2.5, 3.5, 15)
kie_vs_dDA = []

for d in d_DA_values:
    params = {**slo1_params, "d_DA": float(d), "method": "vibronic_multi"}
    r = api_post("/v1/rate/parameters", params)
    kie_vs_dDA.append(r["KIE"])

# Sweep reorganization energy
lambda_values = np.linspace(10, 30, 15)
kie_vs_lambda = []

for lam in lambda_values:
    params = {**slo1_params, "lambda_reorg": float(lam), "method": "vibronic_multi"}
    r = api_post("/v1/rate/parameters", params)
    kie_vs_lambda.append(r["KIE"])

# Plot side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(d_DA_values, kie_vs_dDA, 'o-', color='#2980b9', lw=2, markersize=6)
axes[0].axvline(x=2.7, color='red', linestyle='--', alpha=0.6, label='SLO-1 WT (2.7 A)')
axes[0].set_xlabel('Donor-Acceptor Distance (Angstrom)')
axes[0].set_ylabel('KIE (k_H / k_D)')
axes[0].set_title('KIE vs Donor-Acceptor Distance', fontsize=13, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(lambda_values, kie_vs_lambda, 's-', color='#e74c3c', lw=2, markersize=6)
axes[1].axvline(x=19.0, color='red', linestyle='--', alpha=0.6, label='SLO-1 WT (19 kcal/mol)')
axes[1].set_xlabel('Reorganization Energy (kcal/mol)')
axes[1].set_ylabel('KIE (k_H / k_D)')
axes[1].set_title('KIE vs Reorganization Energy', fontsize=13, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Parameter Sensitivity Analysis (Vibronic Multi-Mode)',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 4. Uncertainty Quantification

Experimental Marcus parameters carry uncertainties from fitting, DFT calculations, or
spectroscopic measurements. The `/v1/rate/uncertainty` endpoint performs **Monte Carlo
propagation** of parameter uncertainties through the rate equations, returning confidence
intervals and parameter sensitivities.

In [ ]:
uncertainty_params = {
    "V_el": 1.0, "V_el_err": 0.3,
    "delta_G": -5.4, "delta_G_err": 1.0,
    "lambda_reorg": 19.0, "lambda_reorg_err": 2.0,
    "omega_H": 2900, "omega_H_err": 100,
    "d_DA": 2.7, "d_DA_err": 0.1,
    "n_samples": 5000,
    "seed": 42
}

uq_result = api_post("/v1/rate/uncertainty", uncertainty_params)

# Display results
print(f"{'Quantity':<15} {'Mean':>12} {'Std':>12} {'95% CI':>25}")
print("=" * 66)
for key, label in [("k_H", "k_H (s^-1)"), ("k_D", "k_D (s^-1)"), ("KIE", "KIE")]:
    mean_key = f"{key}_mean" if f"{key}_mean" in uq_result else "mean"
    std_key = f"{key}_std" if f"{key}_std" in uq_result else "std"
    ci_key = f"{key}_ci95" if f"{key}_ci95" in uq_result else "ci95"

    # Handle both flat and nested response formats
    if f"{key}_mean" in uq_result:
        mean_val = uq_result[f"{key}_mean"]
        std_val = uq_result[f"{key}_std"]
        ci = uq_result[f"{key}_ci95"]
    elif key in uq_result and isinstance(uq_result[key], dict):
        mean_val = uq_result[key]["mean"]
        std_val = uq_result[key]["std"]
        ci = uq_result[key]["ci95"]
    else:
        continue

    if key in ("k_H", "k_D"):
        print(f"{label:<15} {mean_val:>12.2e} {std_val:>12.2e} [{ci[0]:.2e}, {ci[1]:.2e}]")
    else:
        print(f"{label:<15} {mean_val:>12.1f} {std_val:>12.1f} [{ci[0]:.1f}, {ci[1]:.1f}]")

# Sensitivity bar chart
sensitivities = uq_result.get("sensitivities", {})
if sensitivities:
    fig, ax = plt.subplots(figsize=(10, 5))

    param_names = list(sensitivities.keys())
    param_vals = [abs(sensitivities[p]) for p in param_names]

    # Sort by magnitude
    order = np.argsort(param_vals)[::-1]
    param_names = [param_names[i] for i in order]
    param_vals = [param_vals[i] for i in order]

    colors = plt.cm.Blues(np.linspace(0.4, 0.9, len(param_names)))
    ax.barh(range(len(param_names)), param_vals, color=colors,
            edgecolor='black', linewidth=0.5)
    ax.set_yticks(range(len(param_names)))
    ax.set_yticklabels(param_names, fontsize=11)
    ax.set_xlabel('Sensitivity (|d(KIE)/d(param)|)', fontsize=12)
    ax.set_title('KIE Parameter Sensitivity', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='x')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()
else:
    print("\n(Sensitivity data not available in response)")

## 5. Electrochemical PCET

In electrochemical settings (fuel cells, electrolyzers, CO2 reduction), an applied
overpotential shifts the driving force and thus the PCET rate. The `/v1/rate/electrochemical`
endpoint computes rates as a function of overpotential, producing **Tafel plots** that
connect molecular-level PCET theory to measurable electrode kinetics.

In [ ]:
overpotentials = np.linspace(-0.5, 0.5, 21)

k_H_anodic, k_H_cathodic = [], []
k_D_anodic, k_D_cathodic = [], []

base_echem = {
    "V_el": 1.0,
    "delta_G_base": -5.4,
    "lambda_reorg": 19.0,
    "omega_H": 2900,
    "d_DA": 2.7
}

for eta in overpotentials:
    # Anodic (oxidation) direction
    params_a = {**base_echem, "overpotential": float(eta), "direction": "anodic"}
    r_a = api_post("/v1/rate/electrochemical", params_a)
    k_H_anodic.append(r_a["k_H"])
    k_D_anodic.append(r_a["k_D"])

    # Cathodic (reduction) direction
    params_c = {**base_echem, "overpotential": float(eta), "direction": "cathodic"}
    r_c = api_post("/v1/rate/electrochemical", params_c)
    k_H_cathodic.append(r_c["k_H"])
    k_D_cathodic.append(r_c["k_D"])

# Tafel plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: log(k_H) vs overpotential
axes[0].semilogy(overpotentials, k_H_anodic, 'o-', color='#2980b9', lw=2,
                 markersize=5, label='Anodic (H)')
axes[0].semilogy(overpotentials, k_H_cathodic, 's-', color='#e74c3c', lw=2,
                 markersize=5, label='Cathodic (H)')
axes[0].axvline(x=0, color='gray', linestyle='--', alpha=0.5)
axes[0].set_xlabel('Overpotential (V)')
axes[0].set_ylabel('k_H (s^-1)')
axes[0].set_title('Tafel Plot: H Transfer Rate', fontsize=13, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Right: KIE vs overpotential
kie_anodic = [h / d if d > 0 else 0 for h, d in zip(k_H_anodic, k_D_anodic)]
kie_cathodic = [h / d if d > 0 else 0 for h, d in zip(k_H_cathodic, k_D_cathodic)]

axes[1].plot(overpotentials, kie_anodic, 'o-', color='#2980b9', lw=2,
             markersize=5, label='Anodic')
axes[1].plot(overpotentials, kie_cathodic, 's-', color='#e74c3c', lw=2,
             markersize=5, label='Cathodic')
axes[1].axvline(x=0, color='gray', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Overpotential (V)')
axes[1].set_ylabel('KIE (k_H / k_D)')
axes[1].set_title('KIE vs Overpotential', fontsize=13, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Electrochemical PCET: Rate-Overpotential Relationships',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 6. Mutant Comparison

Site-directed mutagenesis of SLO-1 modulates the donor-acceptor distance, providing a
stringent test of PCET theory. Key mutants include:

- **Wild type (WT)**: d_DA = 2.7 A, KIE ~ 80
- **L546A**: Leucine-to-alanine at position 546 increases d_DA to ~2.9 A
- **L754A**: Leucine-to-alanine at position 754 increases d_DA to ~3.0 A

Larger d_DA suppresses tunneling and reduces KIE -- a signature prediction of vibronic theory.

In [ ]:
mutants = {
    "SLO-1 WT":   {"d_DA": 2.7},
    "SLO-1 L546A": {"d_DA": 2.9},
    "SLO-1 L754A": {"d_DA": 3.0}
}

base_params = {
    "V_el": 1.0,
    "delta_G": -5.4,
    "lambda_reorg": 19.0,
    "omega_H": 2900,
    "method": "vibronic_multi",
    "temperature": 303.0
}

mutant_results = {}
for name, overrides in mutants.items():
    params = {**base_params, **overrides}
    mutant_results[name] = api_post("/v1/rate/parameters", params)

# Table
print(f"{'Mutant':<18} {'d_DA (A)':>9} {'k_H (s^-1)':>13} {'k_D (s^-1)':>13} {'KIE':>8}")
print("=" * 63)
for name in mutants:
    r = mutant_results[name]
    d = mutants[name]['d_DA']
    print(f"{name:<18} {d:>9.1f} {r['k_H']:>13.2e} {r['k_D']:>13.2e} {r['KIE']:>8.1f}")

# Bar chart
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

names_list = list(mutants.keys())
kie_vals = [mutant_results[n]["KIE"] for n in names_list]
kH_vals = [mutant_results[n]["k_H"] for n in names_list]

colors = ['#2980b9', '#27ae60', '#e74c3c']

axes[0].bar(names_list, kie_vals, color=colors, edgecolor='black', linewidth=0.5)
axes[0].set_ylabel('KIE (k_H / k_D)')
axes[0].set_title('KIE by SLO-1 Mutant', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='y')

axes[1].bar(names_list, kH_vals, color=colors, edgecolor='black', linewidth=0.5)
axes[1].set_ylabel('k_H (s^-1)')
axes[1].set_title('H Transfer Rate by Mutant', fontsize=13, fontweight='bold')
axes[1].ticklabel_format(style='scientific', axis='y', scilimits=(0,0))
axes[1].grid(True, alpha=0.3, axis='y')

plt.suptitle('SLO-1 Mutant Series: Distance Modulates Tunneling',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"\nKIE drops {kie_vals[0]/kie_vals[-1]:.1f}x from WT to L754A ")
print(f"as d_DA increases by {mutants['SLO-1 L754A']['d_DA'] - mutants['SLO-1 WT']['d_DA']:.1f} A.")
print("This confirms the tunneling-distance relationship predicted by vibronic theory.")

## Key Takeaways

| Feature | pyPCET | DFT (Gaussian/ORCA) | **PCET Engine (Ours)** |
|---------|:------:|:-------------------:|:----------------------:|
| **Setup time** | Hours (install + config) | Days (basis sets + QM) | **Seconds (API call)** |
| **Rate calculation** | Local Python | Local HPC | **Cloud API** |
| **Methods** | Vibronic only | Electronic structure | **Marcus + vibronic + electrochemical** |
| **Uncertainty quantification** | Manual | Manual | **Built-in Monte Carlo** |
| **Benchmarked enzymes** | ~5 | Case-by-case | **15 validated systems** |
| **Electrochemical PCET** | No | No | **Yes (Tafel analysis)** |
| **Mutant prediction** | Manual | Manual | **Direct d_DA modulation** |
| **Batch processing** | Script-level | Job queues | **API-native** |

### API Access

- **Free tier**: 100 requests/month
- **Pro tier**: Unlimited requests, batch processing, priority support
- **Enterprise**: On-premise deployment, custom parameterizations

### Contact

**OmniSciences LLC**  
Email: sloan@omnisciences.org  
Web: [omnisciences.io](https://omnisciences.io)  

*Patent pending. (c) 2026 OmniSciences LLC.*